# The web UI and the API it uses

The calls the React app makes, step by step (needs the stack).

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

In [ ]:
import httpx
from marginalia.config import Settings
s = Settings()
api = httpx.Client(base_url='http://127.0.0.1:8010/api', timeout=30)
login = api.post('/auth/login', json={'email': s.admin_email, 'password': s.admin_password})
print(login.status_code, login.json().get('user'))
api.headers['Authorization'] = f"Bearer {login.json()['access_token']}"

## The shelf, a book, its questions

In [ ]:
books = api.get('/books').json()
for b in books:
    print(b['title'], '|', b['status'], '|', b['cover_url'])
if books:
    qs = api.get(f"/books/{books[0]['id']}/questions").json()
    print([(q['text'], q['status']) for q in qs[:5]])

## Answer settings the UI offers

In [ ]:
api.get("/questions/sampling").json()

## Live events: the websocket
The first message must carry the access token.

In [ ]:
import asyncio, json, websockets
async def listen(seconds=5):
    async with websockets.connect('ws://127.0.0.1:8010/api/ws') as ws:
        await ws.send(json.dumps({'type': 'auth', 'token': api.headers['Authorization'][7:]}))
        try:
            while True:
                print(await asyncio.wait_for(ws.recv(), seconds))
        except asyncio.TimeoutError:
            print('(quiet)')
await listen()